# Final Project — Topic #44: Yelp Reviews at Scale: Ratings, Categories and Cities
**Course:** Programming for Data Analysis and Database  
**Program:** MSc in Applied Artificial Intelligence  
**Cluster:** Cluster 8 — Large-Scale Data Processing (Pandas, Polars, Dask, Parquet)  
**Team Members:** **Uali Sembay** & **Arystan Aldiyev** (`AAI-2605`)  
**Dataset Source:** [Official Yelp Open Dataset (`Yelp-JSON.zip`, 4.35 GB)](https://business.yelp.com/data/resources/open-dataset/)

---
### Executive Summary of Notebook Sections
1. **Out-of-Core Streaming Extraction & Parquet ETL:** Line-by-line JSONL processing of **750,000 real Yelp reviews** (`547.06 MB`), **150,346 real businesses** (`113.36 MB`), and **444,680 user profiles** (`86.88 MB`), totaling **747.30 MB** (`1,345,026` total records).
2. **Data Quality Audit & Memory Optimization:** Auditing duplicate copy-paste spam (`1,273` rows), ultra-short reviews (`13` rows), missing business categories (`103` rows), and extreme vote outliers (`55` rows), achieving **43.3% RAM reduction** and **1.8x Snappy Parquet compression**.
3. **Multi-Engine Benchmarking:** Comparing **JSONL Chunked Baseline** (`68.56s`) vs. **Pandas Parquet** (`1.84s`) vs. **Dask Lazy DataFrame** (`2.68s`) vs. **Polars LazyFrame** (`0.19s`, **361.8x speedup**).
4. **Relational Database (5-Table Normalized SQLite Schema) & 8 Analytical SQL Queries:** Executing `JOIN`, `GROUP BY / HAVING`, `CTE`, subqueries, and window functions (`RANK`, `DENSE_RANK`, `LAG`, `NTILE`, `AVG OVER`).
5. **Exploratory Data Analysis & Statistical Hypothesis Testing:** Testing 3 hypotheses (`Welch's t-test`, `Levene's test`, `One-Way ANOVA`, `Mann-Whitney U`, `Spearman rho`) and rendering static + interactive Plotly charts.
6. **Leakage-Free NLP Star-Rating Prediction:** Training Baseline (`DummyClassifier`) vs. `MultinomialNB`, `SGDClassifier`, and `LogisticRegression` on `TF-IDF (1-2 ngrams)` with a strict chronological split (`Train: 2005–2019`, `Test: 2020–2022`).

In [1]:
import json
import pandas as pd
import polars as pl
from IPython.display import Image, display

from src.config import (
    AUDIT_REPORT_JSON,
    BENCHMARK_RESULTS_JSON,
    HYPOTHESIS_RESULTS_JSON,
    MODEL_METRICS_JSON,
    PLOTS_DIR,
)
from src.database import execute_all_sql_queries
from src.ml_models import predict_custom_review

print('Environment and project modules loaded successfully!')

## 1. Data Quality Audit & Memory Optimization Results

In [2]:
with open(AUDIT_REPORT_JSON, 'r') as f:
    audit_report = json.load(f)

print('=== REVIEWS DATA QUALITY AUDIT ===')
display(pd.DataFrame([audit_report['reviews_audit']]))

print('\n=== MEMORY & DISK OPTIMIZATION ===')
display(pd.DataFrame([audit_report['reviews_memory_optimization']]))

## 2. Large-Scale Engine Benchmark: JSONL Chunked Baseline vs. Pandas vs. Dask vs. Polars

In [3]:
with open(BENCHMARK_RESULTS_JSON, 'r') as f:
    bench_data = json.load(f)

df_bench = pd.DataFrame(bench_data['benchmarks'])
display(df_bench)
display(Image(filename=str(PLOTS_DIR / 'fig3_engine_benchmarks.png')))

## 3. Normalized 5-Table SQLite Database & 8 Analytical SQL Queries

In [4]:
sql_results = execute_all_sql_queries()
for q_name, df_q in sql_results.items():
    print(f'\n================ {q_name} ================')
    display(df_q.head(6))

## 4. Exploratory Data Analysis & Statistical Hypothesis Testing

In [5]:
with open(HYPOTHESIS_RESULTS_JSON, 'r') as f:
    hyp_data = json.load(f)

for key, val in hyp_data.items():
    print(f"*** {val['title']} -> {val['decision']} ***")
    print(f"Conclusion: {val['conclusion']}\n")

display(Image(filename=str(PLOTS_DIR / 'fig1_elite_vs_regular_ratings.png')))
display(Image(filename=str(PLOTS_DIR / 'fig2_city_category_heatmap.png')))
display(Image(filename=str(PLOTS_DIR / 'fig4_yearly_city_trends.png')))
display(Image(filename=str(PLOTS_DIR / 'fig5_iqr_boxplots.png')))

## 5. NLP Star-Rating Prediction from Review Text (Chronological Split) & Error Analysis

In [6]:
with open(MODEL_METRICS_JSON, 'r') as f:
    ml_data = json.load(f)

print('Split Strategy:', ml_data['split_strategy'])
display(pd.DataFrame(ml_data['models_comparison']))
display(Image(filename=str(PLOTS_DIR / 'fig6_confusion_matrix.png')))

print('\n=== SAMPLE MISCLASSIFIED REVIEWS (ERROR ANALYSIS) ===')
display(pd.DataFrame(ml_data['error_analysis_samples']).head(5))

## 6. Interactive Custom Review Prediction Demo

In [7]:
sample_reviews = [
    'Phenomenal dinner in New Orleans! Fresh oysters, attentive waiter, and incredible jazz ambiance.',
    'Waited over an hour for lukewarm pizza and the cashier was extremely rude. Never coming back.',
    'Decent neighborhood coffee shop. Pastries are okay and WiFi works, though prices are a bit average.'
]
for txt in sample_reviews:
    res = predict_custom_review(txt)
    print(f"Review: {txt[:75]}...")
    print(f"  -> Predicted: {res['predicted_stars']} Stars ({res['sentiment_category']}) | Probs: {res['class_probabilities_pct']}\n")